## **Scaling Pandas with Spark**

### **In this lesson you:**
- Demostrate the similarities of the pandas API on Spark API with the pandas API
- Understand teh differences in syntax for the same DataFrame operations in pandas API on Spark vs PySpark 


Recall from the cloud computing lesson the open source technology [Apache Spark](https://spark.apache.org/) , which is an open-source data procesing engine that manages distribuited processing on large data sets.

In this course, we have been using Pandas, which is a great library for data analysis, but it only runs on one machine, its not distribuited. Luckily, we can use Spark just like traditional Pandas with the Pandas with the Pandas API on Spark.

The Pandas API on Spark project makes data scientists more productive when interacting with big data, by implementing the pandas DataFrame API on top of Apache Spark. By unifying the two ecosystems with a familiar API, pandas API on Spark offers a seamless transition betwween small and large data.

See this **blog post** for more information on using Pandas syntax on Spark.

#### **InternalFrame**

The InternalFrame holds the current Spark DataFrame and internal immutable metadata.

It manages mappings from pandas API on Spark column names to Spark column names, as well as from pandas API on Spark index names to Spark column names.

If a user calls some API, the pandas API on Spark DataFrame updates the Spark DataFrame and metadata in InternalFrame. It creates or copies the current InternalFrame with the new states, and returns a new pandas API on Spark DataFrame.

```
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│  Pandas API on  │──────▶│ InternalFrame   │──────▶│     Spark       │
│     Spark       │       │ - column_labels │       │   DataFrame     │
│   DataFrame     │       │ - index_map     │       │                 │
└─────────────────┘       └─────────────────┘       └─────────────────┘
        │                                                    │
        │ API call          copy with new state              │
        │ · · · · · · · · · · · · · · · · ▶                  │
        ▼                                                    ▼
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│  Pandas API on  │──────▶│ InternalFrame   │──────▶│     Spark       │
│     Spark       │       │ - column_labels │       │   DataFrame     │
│   DataFrame     │       │ - index_map     │       │                 │
└─────────────────┘       └─────────────────┘       └─────────────────┘
```

---

#### Read in the dataset
- PySpark
- pandas
- pandas API on Spark

Just like how we can read in data from a CSV file in pandas, we can read in data from a file in spark.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.17/libexec/openjdk.jdk/Contents/Home"


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ITP 13 - Scaling Pandas with Spark") \
    .getOrCreate()

spark_df = spark.read.csv(f"../resources/sf_airbnb_listings.csv", header=True, inferSchema=True, multiLine=True, escape="\"")
display(spark_df)

Read in CSV with pandas

In [ ]:
import pandas as pd
pandas_df = pd.read_csv(f"../resources/sf_airbnb_listings.csv")
pandas_df.head()

Read in CSV with pandas API on Spark. You'll notice pandas API on Spark generates an index column for you, like in pandas.

In [ ]:
import pyspark.pandas as ps
df = ps.read_csv(f"../resources/sf_airbnb_listings.csv", inferSchema=True, multiLine=True, escape="\"")
df.head

#### **Index Types**

- `sequence`
    * Used by default
    * Implement sequence that increase one by one
    * Uses PySpark windows function whitout specifying partition
    * Can end up with whole partition on single node
    * Avoid when data is large
- `distributed-sequence`
    * Implements sequence that increases one by one
    * Uses group-by and group-map in distributed manner
    * Recommended if the index must be sequential for a larga dataset and increasing one by one
- `distributed`
    * Implements monotonically increasing sequence
    * Uses PySpark's _monotonically_incresing_id_ in distributed manner
    * Values are non-deterministic
    * Recommended if the index _does not_ have to be a sequence increasing one by one.

Configurable by the option `compute.defaut_index_type` 

In [ ]:
ps.set_option('compute.default_index_type', 'distributed-sequence')
df_distributed = ps.read_csv(f"../resources/sf_airbnb_listings.csv", inferSchema=True, multiLine=True, escape="\"")
df_distributed.head

####  **Converting to pandas API on Spark DataFrame to/from Spark DataFrame**

Creating a pandas API on Spark DataFrame from PySpark DataFrame.

In [ ]:
df = ps.DataFrame(spark_df)
display(df)

Alternative way of creating a pandas API on Spark DataFrame from PySpark DataFrame

In [ ]:
# df = spark_df.to_pandas_on_spark()
df = spark_df.toPandas()
display(df)

Go from a pandas API on Spark DataFrame to a Spark DataFrame

In [ ]:
display(df.to_spark())